<a href="https://colab.research.google.com/github/honordold/DS1001-LABS-Projects/blob/main/notebooks/02-sql-databases/2026-09-11%20%E2%80%94%20SQL%20Challenge%20Set%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [12]:
q('''
SELECT t.track_id, t.title, t.genre, t.seconds,
       a.name AS artist, a.country
FROM tracks t
LEFT JOIN artists a ON a.artist_id = t.artist_id
ORDER BY t.track_id
''')

,track_id,title,genre,seconds,artist,country
0,10,Skyline,Pop,201,Nova Waves,US
1,11,Undertow,Pop,240,Nova Waves,US
2,12,Foothills,Folk,185,The Blue Ridge,US
3,13,Aurora,Electronic,300,Kestrel,UK
4,14,Nightfall,Electronic,275,Kestrel,UK
5,15,Sol,Latin,210,Marisol,ES
6,16,Coastline,Folk,199,The Blue Ridge,US
7,17,Ridgeline,Folk,225,The Blue Ridge,US
8,18,Untitled Demo,None,150,Kestrel,UK


### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [13]:
q('''
SELECT genre, ROUND(AVG(seconds), 1) AS avg_seconds, COUNT(*) AS n_tracks
FROM tracks
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY avg_seconds DESC
''')

,genre,avg_seconds,n_tracks
0,Electronic,287.5,2
1,Pop,220.5,2
2,Latin,210.0,1
3,Folk,203.0,3


### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [14]:
q('''
SELECT user,
       COUNT(*) AS plays,
       COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY plays DESC
''')

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,dan,2,2
3,cara,2,2


### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [15]:
q('''
SELECT t.track_id, t.title
FROM tracks t
LEFT JOIN plays p ON p.track_id = t.track_id
WHERE p.play_id IS NULL
ORDER BY t.track_id
''')

,track_id,title
0,17,Ridgeline
1,18,Untitled Demo


### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [16]:
q('''
SELECT a.name AS artist,
       COUNT(p.play_id) AS plays,
       SUM(t.seconds) AS total_seconds,
       ROUND(SUM(t.seconds) / 60.0, 1) AS total_minutes
FROM plays p
JOIN tracks t ON t.track_id = p.track_id
JOIN artists a ON a.artist_id = t.artist_id
GROUP BY a.artist_id, a.name
ORDER BY total_seconds DESC
''')

,artist,plays,total_seconds,total_minutes
0,Kestrel,4,1175,19.6
1,Nova Waves,4,843,14.1
2,The Blue Ridge,2,384,6.4
3,Marisol,1,210,3.5


### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [17]:
q('''
SELECT track_id, title
FROM tracks
WHERE genre IS NULL
-- WHERE genre != 'Pop' would NOT return this row. Comparing NULL to
-- 'Pop' yields NULL, not TRUE, and WHERE only keeps rows that evaluate
-- to TRUE. So the untagged track gets silently dropped from any
-- inequality filter. NULL means "unknown", and unknown != 'Pop' is
-- itself unknown. That is why IS NULL exists as separate syntax.
''')

,track_id,title
0,18,Untitled Demo


### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [18]:
q('''
SELECT played_on AS play_date,
       COUNT(*) AS plays,
       COUNT(DISTINCT user) AS active_users
FROM plays
GROUP BY played_on
ORDER BY played_on
''')

,play_date,plays,active_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [19]:
# assert len(q1) == 9, 'Q1 should return one row per track'
# assert len(q4) == 2, 'Q4: two tracks have never been played'
# assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

_Q4 gave me the most trouble. My first version used JOIN instead of LEFT JOIN, and it returned zero rows. The inner join only kept tracks that had a match in plays, so the two unplayed tracks were already gone before the WHERE p.play_id IS NULL filter ran. Switching to LEFT JOIN keeps all 9 tracks and fills the plays columns with NULL for the unmatched ones, which is what makes the filter work. Ridgeline and Untitled Demo come back as expected. The lesson was that zero rows is a result I need to check, not a non-answer. I only caught it because the prompt said to expect 2._